In [1]:
import os
import sys

import numpy as np
import pandas as pd
import plotly.express as px
import dataframe_image as dfi
import plotly.graph_objects as go
from scipy import stats 
import sqlalchemy as db

sys.path.append(os.path.abspath(os.path.join('../..')))

from utils import layout, get_sample_grid

In [2]:
username = "amos"
password = "M0$hicat"
host = "192.168.0.131"
port = "3306"
database = "CineFaceDW"
connection_string = f'mysql+pymysql://{username}:{password}@{host}:{port}/{database}'
engine = db.create_engine(connection_string)
conn = engine.connect()

In [3]:
name = "Sergio Leone"

In [4]:
def create_gridmap_from_director(director, 
                                 engine, 
                                 layout=None, 
                                 width=600,
                                 height=600,
                                 transparent=False,
                                 dst=None):
    with engine.connect() as conn:
        query = "SELECT * FROM vwFacesByDirector WHERE name = :director"
        df = pd.read_sql_query(db.text(query), conn, params={"director": director})

    if df.empty:
        print(f"Could not find {director} in database.")
        return

    # --- INTERNAL LOGIC: Normalized Face Processing ---
    # We use a 1000x1000 internal grid to normalize all aspect ratios
    res = 1000
    norm_mask = np.zeros(shape=(res, res), dtype=int)
    
    # Calculate normalized centers (0.0 to 1.0) and map to our grid
    cx = (((df['x1'] + df['x2']) / 2) / df['img_width'] * (res - 1)).astype(int).values
    cy = (((df['y1'] + df['y2']) / 2) / df['img_height'] * (res - 1)).astype(int).values
    
    # Filter valid coordinates inside our 1000x1000 space
    valid = (cx >= 0) & (cx < res) & (cy >= 0) & (cy < res)
    np.add.at(norm_mask, (cy[valid], cx[valid]), 1)
    
    # --- INTERNAL LOGIC: Create 3x3 Grid ---
    # Split the 1000x1000 mask into 9 sectors
    h_chunk = res // 3
    w_chunk = res // 3
    grid = np.zeros((3, 3))
    
    total_faces = norm_mask.sum()
    if total_faces > 0:
        for r in range(3):
            for c in range(3):
                sector = norm_mask[r*h_chunk:(r+1)*h_chunk, c*w_chunk:(c+1)*w_chunk]
                grid[r, c] = (sector.sum() / total_faces) * 100
    
    # --- PLOTLY RENDERING ---
    fig = px.imshow(np.round(grid, 3),
                    x=['Left', 'Middle', 'Right'],
                    y=['Top', 'Center', 'Bottom'],
                    labels={'color': '% of Total Faces'},
                    text_auto=".2f",
                    aspect='auto',
                    color_continuous_scale='Aggrnyl')
    
    if layout:
        fig.update_layout(layout)
        
    fig.update_layout(
        title={
            "text": f'Normalized Face Locations: {director}',
            "x": 0.5, "xanchor": "center", "yanchor": "top"
        },
        width=width, height=height,
        xaxis=dict(side="bottom"),
        yaxis=dict(autorange="reversed"), 
        margin=dict(l=50, r=50, t=100, b=50),
        coloraxis_colorbar=dict(title="%"),
        xaxis_title="Horizontal Position",
        yaxis_title="Vertical Position",
        template="plotly_dark"
    )
    fig.show()
    if dst:
        if transparent:
            fig.update_layout(
                paper_bgcolor='rgba(0,0,0,0)', # Transparent outer background
                plot_bgcolor='rgba(0,0,0,0)',  # Transparent inner plot area
                font=dict(color="white")       # Good if your Canva theme is dark
            )
        fig.write_image(dst, width=width, height=height, scale=2)

In [6]:
def plot_grid(grid, plot_title=None, width=600, height=600, layout=None, dst=None, transparent=False):
    # Center the color scale at zero
    limit = max(abs(grid.min()), abs(grid.max()))
    
    fig = px.imshow(
        grid,
        x = ['Left', 'Center', 'Right'],
        y = ['Top', 'Middle', 'Bottom'],
        color_continuous_scale='Aggrnyl', 
        # range_color=[-limit, limit],
        text_auto=".3f", # This shows the difference value in each box
        title=plot_title if plot_title else "",
        aspect="equal"
    )

    if layout:
        fig.update_layout(layout)

    unified_layout = {
        "title": {
            "text": plot_title if plot_title else "",
            "x": 0.5, "xanchor": "center", "yanchor": "top"
        },
        "width": width,
        "height": height,
        "xaxis": dict(side="bottom", title="Horizontal Position"),
        "yaxis": dict(autorange="reversed", title="Vertical Position"), 
        "margin": dict(l=50, r=50, t=100, b=50), # Increased top margin for Canva title safety
        "coloraxis_colorbar": dict(title="%")
    }

    fig.update_layout(unified_layout)
    
    fig.show()
    if dst:
        if transparent:
            fig.update_layout(
                paper_bgcolor='rgba(0,0,0,0)', # Transparent outer background
                plot_bgcolor='rgba(0,0,0,0)',  # Transparent inner plot area
                font=dict(color="white")       # Good if your Canva theme is dark
            )
        fig.write_image(dst, width=width, height=height, scale=2)

In [7]:
def compare_director_to_sample_grid(name, 
                                    engine, 
                                    title=None, 
                                    layout=None,
                                    dst=None,
                                    transparent=False,
                                    width=600,
                                    height=600):
    with engine.connect() as conn:
        query = f"""
            SELECT 
                AVG(pct_tl), AVG(pct_tc) as tc, AVG(pct_tr) as tr,
                AVG(pct_ml), AVG(pct_mc) as mc, AVG(pct_mr) as mr,
                AVG(pct_bl), AVG(pct_bc) as bc, AVG(pct_br) as br
            FROM CineFaceDW.vwGridByJob gbj
            WHERE gbj.name = '{name}' AND gbj.job_name = 'Director'
            GROUP BY gbj.name
            """
        df = pd.read_sql_query(db.text(query), conn)

        
    temp = df.mean()
    grid = temp.values.reshape(3, 3)
    grid_norm = get_sample_grid(engine)
    diff_grid = (grid - grid_norm) * 100
    plot_grid(diff_grid, plot_title=title, layout=layout, dst=dst, width=width, height=height, transparent=transparent)

In [8]:
create_gridmap_from_director(name, engine, dst="./plots/leone/leone_gridmap.png", layout=layout)

In [9]:
compare_director_to_sample_grid(name, engine, layout=layout, title="Sergio Leone vs. Industry Baseline", dst="./plots/leone/leone_vs_industry_baseline.png")